# Day 27 — 개별·군 수준 시각화: 사람이 보이는 그림 그리기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-4/day-27.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-4/day-27.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1: 그림 두 장을 한 번에

가상 feature table을 코드 안에서 만들어 그립니다. Day 26의 CSV가 있으면 표 만드는 부분을 `pd.read_csv`로 바꾸면 됩니다. (그림 속 글자는 한글 글꼴 설정 문제를 피하려고 영어로 적었습니다.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- 가상 feature table (Day 26 파일이 있으면 그 파일을 읽어도 된다) ----------
rng = np.random.default_rng(3)
ids = [f"S{i:02d}" for i in range(1, 13)]
group = ["PD"] * 6 + ["HC"] * 6
base = np.where(np.array(group) == "PD", rng.uniform(0.015, 0.045, 12), rng.uniform(0.009, 0.014, 12))
rows = []
for pid, g, b in zip(ids, group, base):
    for task, scale in [("Relaxed", 1.0), ("StretchHold", 0.7), ("TouchNose", 1.6)]:
        rows.append({"participant_id": pid, "group": g, "task": task,
                     "std_g": b * scale + rng.normal(0, 0.001)})
df = pd.DataFrame(rows)
# ↑↑↑ 실제 데이터로 바꿀 때: df = pd.read_csv("<내 feature table 경로>") 후 열 이름 맞추기 ↑↑↑

tasks = ["Relaxed", "StretchHold", "TouchNose"]
colors = {"PD": "tab:red", "HC": "tab:blue"}

# ---------- Figure 1: spaghetti plot (한 사람 = 선 하나) ----------
fig, ax = plt.subplots(figsize=(6, 4))
for pid, sub in df.groupby("participant_id"):
    sub = sub.set_index("task").loc[tasks]
    ax.plot(tasks, sub["std_g"], marker="o", alpha=0.7, color=colors[sub["group"].iloc[0]])
for g, c in colors.items():
    ax.plot([], [], color=c, marker="o", label=f"{g} (n={df[df.group == g].participant_id.nunique()})")
ax.set_xlabel("Task")
ax.set_ylabel("Movement SD (g)")
ax.set_title("Simulated data: one line per participant")
ax.legend()
fig.tight_layout()
fig.savefig("fig_spaghetti.png", dpi=150)

# ---------- Figure 2: box plot + 개별 점 (한 사람 = 점 하나) ----------
one = df[df["task"] == "Relaxed"]                 # 과제 하나만 골라 '한 사람당 한 값'으로
assert one["participant_id"].is_unique
fig, ax = plt.subplots(figsize=(4.5, 4))
data = [one[one.group == g]["std_g"].values for g in ["HC", "PD"]]
ax.boxplot(data, showfliers=False)
ax.set_xticks([1, 2], ["HC", "PD"])
for i, (g, vals) in enumerate(zip(["HC", "PD"], data), start=1):
    jitter = rng.uniform(-0.08, 0.08, len(vals))  # 점이 겹치지 않게 좌우로 살짝 흩기
    ax.scatter(np.full(len(vals), i) + jitter, vals, color=colors[g], zorder=3)
ax.set_ylabel("Movement SD at rest (g)")
ax.set_title("Simulated data: one dot per participant")
fig.tight_layout()
fig.savefig("fig_box.png", dpi=150)

print(one.groupby("group")["std_g"].describe()[["count", "mean", "50%", "min", "max"]].round(4).to_string())
print("저장 완료: fig_spaghetti.png, fig_box.png")

### 예시 2: box plot이 숨기는 것을 violin plot이 보여 준다

가상의 PD군 안에 "움직임이 작은 사람들"과 "큰 사람들" 두 덩어리가 있다고 해 봅시다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
hc = rng.normal(0.012, 0.002, 40)
pd_ = np.concatenate([rng.normal(0.015, 0.003, 25), rng.normal(0.045, 0.005, 15)])  # 두 덩어리

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)
axes[0].boxplot([hc, pd_]); axes[0].set_xticks([1, 2], ["HC", "PD"]); axes[0].set_title("Box plot")
axes[1].violinplot([hc, pd_], showmedians=True); axes[1].set_xticks([1, 2], ["HC", "PD"]); axes[1].set_title("Violin plot")
axes[0].set_ylabel("Movement SD (g)")
fig.suptitle("Simulated data")
fig.tight_layout()
fig.savefig("fig_box_vs_violin.png", dpi=150)

print("PD 중앙값:", round(float(np.median(pd_)), 4), " PD 평균:", round(float(pd_.mean()), 4))

## 직접 해보기

1. `notebooks/05_pd_feature_analysis.ipynb`에 "Day 27 시각화" 제목 셀을 추가하고, 예시 1을 실행해 그림 두 장이 보이는지 확인합니다.
2. **내 feature table 읽기.** Day 26에서 저장한 표를 읽습니다. spaghetti plot에는 "사람 × 과제(또는 걷기 조건)" 형태의 긴 표가, box plot에는 "한 사람 = 한 줄" 표가 필요합니다.

In [ ]:
df = pd.read_csv("<Day 26에서 저장한 feature table 경로>")   # 자리표시자
print(df.columns.tolist())
print(df["<참가자 ID 열>"].nunique(), "명 /", len(df), "줄")